# 🤖 LLM Fine-Tuning Roadmap — Beginner to Advanced
> **By Bishal Paul** | Computer Science Graduate | Aspiring ML Engineer

---

## 📚 Roadmap Overview

| Stage | Title | Level |
|-------|-------|-------|
| Stage 1 | Foundations (Transformers, Tokenization, Embeddings) | 🟢 Beginner |
| Stage 2 | Basic Fine-Tuning (Classification, NER, QA) | 🟡 Intermediate |
| Stage 3 | Instruction Fine-Tuning (Chatbot, Bangla Assistant) | 🟠 Advanced |
| Stage 4 | PEFT — LoRA & QLoRA | 🔴 Expert |


---
# 🧱 STAGE 1: Foundations (Beginner)
### Topics: Transformers · Tokenization · Embeddings · Pre-training vs Fine-tuning


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install transformers datasets torch -q

### 1.1 — Tokenization (BPE / WordPiece)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

text = 'Hello! I am learning LLM fine-tuning step by step.'
tokens = tokenizer(text, return_tensors='pt')

print('Input IDs     :', tokens['input_ids'])
print('Attention Mask:', tokens['attention_mask'])
print('Decoded tokens:', tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))

### 1.2 — Embeddings & Hidden States


In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained('distilbert-base-uncased')

with torch.no_grad():
    output = model(**tokens)

hidden_states = output.last_hidden_state
print('Hidden state shape:', hidden_states.shape)
# → [batch_size, sequence_length, hidden_dim]
# →  [1, 12, 768]  means: 1 sentence, 12 tokens, 768-dim embedding

# CLS token embedding (sentence representation)
cls_embedding = hidden_states[:, 0, :]
print('CLS embedding shape:', cls_embedding.shape)

### 1.3 — Self-Attention Visualization (concept)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate attention weights for illustration
np.random.seed(42)
words = ['The', 'cat', 'sat', 'on', 'the', 'mat']
n = len(words)
attn = np.random.dirichlet(np.ones(n), size=n)  # rows sum to 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn, cmap='YlOrRd')
ax.set_xticks(range(n)); ax.set_xticklabels(words, rotation=45)
ax.set_yticks(range(n)); ax.set_yticklabels(words)
ax.set_title('Self-Attention Weight Matrix (simulated)', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---
# ⚙️ STAGE 2: Basic Fine-Tuning (Intermediate)
### Sentiment Analysis on IMDB using BERT + Trainer API


In [ ]:
!pip install transformers datasets evaluate accelerate -q

### 2.1 — Load & Preprocess Dataset


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset('imdb')
print(dataset)
print('Sample:', dataset['train'][0])

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

tokenized = dataset.map(tokenize, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Use small subset for quick training (remove slice for full training)
train_dataset = tokenized['train'].shuffle(seed=42).select(range(2000))
eval_dataset  = tokenized['test'].select(range(500))

print('Train size:', len(train_dataset))
print('Eval size :', len(eval_dataset))

### 2.2 — Fine-Tune with Trainer API


In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir='./stage2_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

### 2.3 — Evaluate & Predict


In [ ]:
results = trainer.evaluate()
print('Evaluation Results:', results)

# Inference on custom text
import torch

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=1).item()
    label_map = {0: '😞 Negative', 1: '😊 Positive'}
    return label_map[pred]

print(predict_sentiment('This movie was absolutely fantastic!'))
print(predict_sentiment('Terrible film, complete waste of time.'))

---
# 💬 STAGE 3: Instruction Fine-Tuning (Advanced)
### Project: Bangla ChatGPT Assistant using Mistral-7B


In [ ]:
!pip install transformers datasets trl accelerate bitsandbytes peft -q

### 3.1 — Build Bangla Instruction Dataset


In [ ]:
import json
from datasets import Dataset

# Sample Bangla instruction data (expand with real data for production)
bangla_data = [
    {
        'instruction': 'বাংলাদেশের রাজধানী কী?',
        'input': '',
        'output': 'বাংলাদেশের রাজধানী ঢাকা।'
    },
    {
        'instruction': 'পদ্মা সেতু সম্পর্কে বলো।',
        'input': '',
        'output': 'পদ্মা সেতু বাংলাদেশের একটি বহুমুখী সড়ক ও রেল সেতু। এটি পদ্মা নদীর উপর নির্মিত এবং ২০২২ সালে উদ্বোধন হয়।'
    },
    {
        'instruction': 'Python কী?',
        'input': '',
        'output': 'Python একটি উচ্চ-স্তরের প্রোগ্রামিং ভাষা যা সহজ সিনট্যাক্স এবং বহুমুখী ব্যবহারের জন্য বিখ্যাত।'
    },
]

# Save to JSON
with open('bangla_instruct.json', 'w', encoding='utf-8') as f:
    json.dump(bangla_data, f, ensure_ascii=False, indent=2)

print(f'Dataset created with {len(bangla_data)} samples')
print(json.dumps(bangla_data[0], ensure_ascii=False, indent=2))

### 3.2 — Format Prompt (Alpaca Template)


In [ ]:
ALPACA_PROMPT = """Below is an instruction in Bengali. Write a response in Bengali.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_prompt(example):
    return {
        'text': ALPACA_PROMPT.format(
            instruction=example['instruction'],
            input=example.get('input', ''),
            output=example['output']
        )
    }

dataset = Dataset.from_list(bangla_data)
dataset = dataset.map(format_prompt)

print('Sample formatted prompt:')
print(dataset[0]['text'])

### 3.3 — Fine-Tune Mistral-7B with SFTTrainer (requires GPU)


In [ ]:
# NOTE: Requires a GPU with at least 16GB VRAM (e.g. A100, RTX 3090)
# Use Google Colab Pro or Kaggle GPU for free access

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = 'mistralai/Mistral-7B-v0.1'  # or 'meta-llama/Meta-Llama-3-8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype='bfloat16',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

sft_config = SFTConfig(
    output_dir='./bangla_chatgpt',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    max_seq_length=512,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=sft_config,
)

trainer.train()
trainer.save_model('./bangla_chatgpt_final')
print('✅ Bangla ChatGPT saved!')

---
# 🚀 STAGE 4: PEFT — LoRA & QLoRA (Expert)
### Research: Comparative Analysis of LoRA vs QLoRA


### 4.1 — What is LoRA? (Theory)


In [ ]:
# LoRA decomposes weight update ΔW into two low-rank matrices:
#   ΔW = A × B   where A ∈ R^(d×r), B ∈ R^(r×k), rank r << d,k
#
# Instead of training all W (millions of params),
# we only train A and B (thousands of params).

import numpy as np

d, k, r = 768, 768, 16  # hidden dim, hidden dim, rank

full_params = d * k
lora_params = d * r + r * k
reduction   = full_params / lora_params

print(f'Full weight matrix params : {full_params:,}')
print(f'LoRA params (rank={r})    : {lora_params:,}')
print(f'Parameter reduction       : {reduction:.1f}x fewer params!')

### 4.2 — QLoRA Setup (4-bit Quantization + LoRA)


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype='bfloat16',   # compute in bf16 for speed
    bnb_4bit_use_double_quant=True,      # quantize the quantization constants
)

model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-v0.1',
    quantization_config=bnb_config,
    device_map='auto',
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params: ~8M | all params: ~7B | trainable%: ~0.1%

### 4.3 — LoRA vs QLoRA Benchmark Comparison


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods = ['Full Fine-tuning', 'LoRA (fp16)', 'QLoRA (4-bit)']

# Approximate benchmarks for 7B model (Mistral-7B)
gpu_vram_gb   = [80,  18,  10 ]   # GPU memory required
train_time_hr = [8.0, 3.5, 4.2]   # hours per epoch
accuracy      = [91,  90,  89 ]   # % accuracy on downstream task
trainable_pct = [100, 0.5, 0.13]  # % of trainable params

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#E74C3C', '#3498DB', '#2ECC71']

axes[0].bar(methods, gpu_vram_gb, color=colors)
axes[0].set_title('GPU VRAM Required (GB)', fontsize=11)
axes[0].set_ylabel('VRAM (GB)')

axes[1].bar(methods, accuracy, color=colors)
axes[1].set_ylim(85, 93)
axes[1].set_title('Downstream Accuracy (%)', fontsize=11)
axes[1].set_ylabel('Accuracy %')

axes[2].bar(methods, trainable_pct, color=colors, log=True)
axes[2].set_title('Trainable Parameters (%)', fontsize=11)
axes[2].set_ylabel('% Params (log scale)')

for ax in axes:
    ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=9)

plt.suptitle('LoRA vs QLoRA vs Full Fine-tuning — Research Comparison', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('lora_qlora_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as lora_qlora_comparison.png')

### 4.4 — Save & Merge LoRA Adapter


In [ ]:
# Save only the LoRA adapter weights (very small, ~50MB vs 14GB for full model)
model.save_pretrained('./lora_adapter')
print('LoRA adapter saved!')

# Load and merge for deployment
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1')
merged = PeftModel.from_pretrained(base, './lora_adapter')
merged = merged.merge_and_unload()  # merge LoRA into base weights
merged.save_pretrained('./merged_model')
print('Merged model saved and ready for deployment!')

### 4.5 — Inference with Fine-Tuned Model


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained('./merged_model')
model = AutoModelForCausalLM.from_pretrained('./merged_model', device_map='auto')

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)

prompt = """### Instruction:
বাংলাদেশের মুক্তিযুদ্ধ সম্পর্কে বলো।

### Response:
"""

output = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
print(output[0]['generated_text'])

---
## 🎯 Summary

| Stage | Key Skill Gained | Model | GPU Needed |
|-------|-----------------|-------|------------|
| 1 — Foundations | Tokenization, embeddings, attention | DistilBERT | ❌ CPU OK |
| 2 — Basic Fine-tuning | Trainer API, classification | BERT | ⚡ Free Colab |
| 3 — Instruction Tuning | SFTTrainer, chat format | Mistral-7B | 🔥 16GB GPU |
| 4 — PEFT/QLoRA | LoRA, 4-bit quant, merge | Mistral-7B | ✅ 10GB GPU |

### Next Steps
- Upload your fine-tuned model to **Hugging Face Hub**
- Deploy with **Gradio** or **FastAPI**
- Evaluate with **BLEU, ROUGE, Perplexity**
- Write a research paper comparing LoRA vs QLoRA results
